# 차대차 카테고리 YOLO 전처리/Qwen 비교 실험

작성일자: 2026-07-15

대상 카테고리: `차대차`  
Drive 필터 prefix: `TS_차대차_영상_`

흐름:

1. 해당 카테고리 데이터 로드
2. OpenCV 전처리 적용
3. YOLO 5개 모델 bbox 비교
4. Qwen2.5-VL에 bbox 프레임 전달
5. 결과 CSV/contact sheet 저장


In [ ]:
from pathlib import Path
import csv
import json
import math
import os
import random
import shutil
import subprocess
import sys
from collections import Counter, defaultdict

from IPython.display import Image, display

# 실행 위치가 달라도 프로젝트를 찾으며, 필요하면 VISION_PROJECT_ROOT로 명시한다.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
if current.is_dir():
    candidates.extend(path for path in current.iterdir() if path.is_dir())
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'ai/vision').is_dir() and (path / 'storage').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    configured_root = os.getenv('VISION_PROJECT_ROOT')
    if not configured_root:
        raise FileNotFoundError('Project root not found. Set VISION_PROJECT_ROOT.')
    PROJECT_ROOT = Path(configured_root).expanduser().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai.vision.category_vlm_config import get_category_config, load_experiment_config
from ai.vision.adaptive_preprocessing import enhance_frame_adaptive, select_collision_aware_frames

CATEGORY_KEY = 'car_vs_car'
CATEGORY = get_category_config(CATEGORY_KEY)
EXPERIMENT = load_experiment_config(CATEGORY_KEY)
CATEGORY_LABEL = CATEGORY.label
CATEGORY_PREFIX = CATEGORY.prefix

RAW_BASE_DIR = PROJECT_ROOT / 'storage/vision/datasets/classification/raw_videos'
DRIVE_LISTING = PROJECT_ROOT / 'storage/vision/manifests/drive_listing_aihub.json'
CATEGORY_DOWNLOAD_MANIFEST = PROJECT_ROOT / 'storage/vision/datasets/classification/manifests' / f'{CATEGORY_KEY}_download_manifest.csv'
CATEGORY_RAW_DIR = RAW_BASE_DIR / CATEGORY_LABEL
OUTPUT_DIR = PROJECT_ROOT / 'storage/vision/outputs/category_yolo_qwen_compare' / CATEGORY_KEY / 'known_label_adaptive_32frames'
FRAME_DIR = OUTPUT_DIR / f'frames_{EXPERIMENT.frame_count}'
PREPROCESS_DIR = OUTPUT_DIR / 'preprocessed_frames'
CONTACT_DIR = OUTPUT_DIR / 'contact_sheets'
SUMMARY_CSV = OUTPUT_DIR / 'yolo_summary.csv'
DETAIL_CSV = OUTPUT_DIR / 'yolo_details.csv'
QWEN_OUTPUT_CSV = OUTPUT_DIR / 'qwen_yolo_compare_results.csv'

DRIVE_FOLDER_URL = EXPERIMENT.drive_folder_url
RUN_GDOWN_DOWNLOAD = EXPERIMENT.run_gdown_download
MAX_VIDEOS = EXPERIMENT.max_videos
ANALYSIS_SAMPLE_COUNT = EXPERIMENT.analysis_sample_count
FRAME_COUNT = EXPERIMENT.frame_count
QWEN_INPUT_FRAME_COUNT = EXPERIMENT.vlm_input_frame_count
YOLO_CONF = EXPERIMENT.yolo_conf
YOLO_IMGSZ = EXPERIMENT.yolo_imgsz
YOLO_MODELS = list(EXPERIMENT.yolo_models)
RUN_MODEL_COMPARISON = EXPERIMENT.run_model_comparison
FORCE_PREPROCESS = EXPERIMENT.force_preprocess
RELEVANT_CLASSES = {'person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck'}

for path in [CATEGORY_RAW_DIR, OUTPUT_DIR, FRAME_DIR, PREPROCESS_DIR, CONTACT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CATEGORY_LABEL:', CATEGORY_LABEL)
print('CATEGORY_RAW_DIR:', CATEGORY_RAW_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('ANALYSIS_SAMPLE_COUNT:', ANALYSIS_SAMPLE_COUNT)


In [ ]:
def run_command(command, timeout=None, enabled=True):
    print('$', ' '.join(map(str, command)))
    if not enabled:
        print('SKIPPED')
        return None
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, timeout=timeout)
    completed.check_returncode()
    return completed

RUN_INSTALL_REQUIREMENTS = True
if RUN_INSTALL_REQUIREMENTS:
    run_command([sys.executable, '-m', 'pip', 'install', 'ultralytics', 'opencv-python-headless', 'Pillow', 'tqdm', 'transformers==4.51.3', 'accelerate', 'qwen-vl-utils[decord]'], timeout=1800)


In [ ]:
import cv2
import numpy as np
from PIL import Image as PILImage, ImageDraw
from tqdm.auto import tqdm
from ultralytics import YOLO

random.seed()


## 데이터 로드

기존 `raw_videos/{카테고리}` 폴더에 mp4가 있으면 재사용한다. 없고 `RUN_GDOWN_DOWNLOAD=True`이면 `DRIVE_FOLDER_URL`에서 다운로드를 시도한다.

다운로드 후에는 `CATEGORY_PREFIX`로 시작하는 Train 하위 폴더 또는 기존 라벨 폴더에서 mp4를 수집한다.


In [ ]:
def ensure_classification_manifest():
    manifest = PROJECT_ROOT / 'storage/vision/datasets/classification/manifests/classification_manifest.csv'
    manifest.parent.mkdir(parents=True, exist_ok=True)
    if manifest.exists():
        return manifest
    if not DRIVE_LISTING.exists():
        raise FileNotFoundError(f'drive_listing_aihub.json이 필요합니다: {DRIVE_LISTING}')
    run_command([
        sys.executable,
        'etl/vision/build_classification_manifest.py',
        '--listing', DRIVE_LISTING,
        '--output', manifest,
    ], timeout=None)
    return manifest


def read_csv(path):
    with Path(path).open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def write_csv(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(key for row in rows for key in row.keys()))
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    with temporary_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary_path.replace(path)


def category_manifest_rows():
    manifest = ensure_classification_manifest()
    rows = []
    for row in read_csv(manifest):
        if row.get('input_type') != 'video':
            continue
        text = f"{row.get('category', '')}/{row.get('source_path', '')}".replace('*', '_')
        if CATEGORY_PREFIX in text or (f'TS_{CATEGORY_LABEL}' in text and '영상' in text):
            rows.append(row)
    return rows


def category_video_paths():
    paths = []
    if CATEGORY_RAW_DIR.exists():
        paths.extend(CATEGORY_RAW_DIR.rglob('*.mp4'))

    for folder in RAW_BASE_DIR.rglob('*') if RAW_BASE_DIR.exists() else []:
        if folder.is_dir():
            text = folder.name.replace('*', '_')
            if CATEGORY_PREFIX in text or (f'TS_{CATEGORY_LABEL}' in text and '영상' in text):
                paths.extend(folder.rglob('*.mp4'))

    unique = sorted({p.resolve() for p in paths})
    if MAX_VIDEOS is not None:
        unique = unique[:MAX_VIDEOS]
    return unique


def download_category_from_manifest():
    rows = category_manifest_rows()
    print('category_manifest_video_rows:', len(rows))
    if not rows:
        return
    if MAX_VIDEOS is not None:
        rows = rows[:MAX_VIDEOS]
    write_csv(rows, CATEGORY_DOWNLOAD_MANIFEST)
    run_command([
        sys.executable,
        'etl/vision/download_sampled_media.py',
        '--input', CATEGORY_DOWNLOAD_MANIFEST,
        '--output', CATEGORY_DOWNLOAD_MANIFEST,
        '--download-dir', RAW_BASE_DIR,
        '--label-column', 'coarse_label',
        '--per-label', str(len(rows)),
        '--split', '',
    ], timeout=None)


classification_manifest = PROJECT_ROOT / 'storage/vision/datasets/classification/manifests/classification_manifest.csv'
manifest_rows = []
if classification_manifest.exists() or DRIVE_LISTING.exists():
    manifest_rows = category_manifest_rows()

target_count = len(manifest_rows)
if MAX_VIDEOS is not None and target_count:
    target_count = min(target_count, MAX_VIDEOS)

paths = category_video_paths()
print('existing_video_count_before_download:', len(paths))
print('manifest_category_video_rows:', len(manifest_rows))

if manifest_rows and len(paths) < target_count:
    print(f'local files are incomplete: {len(paths)} / {target_count}. downloading missing category videos...')
    download_category_from_manifest()
    paths = category_video_paths()
elif not paths:
    download_category_from_manifest()
    paths = category_video_paths()

if not paths:
    raise FileNotFoundError(f'{CATEGORY_LABEL} mp4? ?? ?????: {RAW_BASE_DIR}')

print('video_count:', len(paths))
print('MAX_VIDEOS:', MAX_VIDEOS)
for p in paths[:5]:
    print(p)


In [ ]:
def selected_video_paths(paths):
    candidates = list(paths)
    random.shuffle(candidates)
    if ANALYSIS_SAMPLE_COUNT is None:
        return candidates
    return candidates[:ANALYSIS_SAMPLE_COUNT]

selected_paths = selected_video_paths(paths)
print('selected_count:', len(selected_paths))
for p in selected_paths:
    print(p.name)


## OpenCV 전처리 및 프레임 추출

100프레임을 추출한 뒤 원본 프레임과 전처리 프레임을 모두 저장한다. VLM에는 설정된 수의 대표 프레임만 균등 샘플링해 전달한다.


In [ ]:
def video_duration_sec(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0.0
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    total = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    cap.release()
    return float(total / fps) if fps else 0.0


def enhance_frame(frame):
    return enhance_frame_adaptive(frame)


def lighting_from_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    mean = float(gray.mean())
    if mean < 45:
        return 'night', mean
    if mean < 85:
        return 'low_light', mean
    return 'day', mean


def extract_frames(video_path, frame_dir, preprocess_dir, frame_count):
    frame_dir.mkdir(parents=True, exist_ok=True)
    preprocess_dir.mkdir(parents=True, exist_ok=True)
    cached_raw = sorted(frame_dir.glob('frame_*.jpg'))
    cached_prep = sorted(preprocess_dir.glob('frame_*.jpg'))
    if not FORCE_PREPROCESS and len(cached_raw) == frame_count and len(cached_prep) == frame_count:
        return cached_raw, cached_prep
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video: {video_path}')
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    if total <= 0:
        cap.release()
        raise RuntimeError(f'Invalid video: {video_path}')

    indices = [round(i * (total - 1) / max(frame_count - 1, 1)) for i in range(frame_count)]
    raw_paths, prep_paths = [], []
    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            continue
        sec = idx / fps if fps else 0
        raw_path = frame_dir / f'frame_{i:03d}_{sec:06.2f}s.jpg'
        prep_path = preprocess_dir / f'frame_{i:03d}_{sec:06.2f}s.jpg'
        cv2.imwrite(str(raw_path), frame)
        cv2.imwrite(str(prep_path), enhance_frame(frame))
        raw_paths.append(raw_path)
        prep_paths.append(prep_path)
    cap.release()
    return raw_paths, prep_paths


video_rows = []
skipped_videos = []
for path in tqdm(selected_paths, desc='extract/preprocess'):
    asset_id = path.stem
    raw_frame_dir = FRAME_DIR / asset_id
    prep_frame_dir = PREPROCESS_DIR / asset_id
    try:
        raw_frames, prep_frames = extract_frames(path, raw_frame_dir, prep_frame_dir, FRAME_COUNT)
    except RuntimeError as exc:
        skipped_videos.append({'asset_id': asset_id, 'local_path': path.as_posix(), 'error': str(exc)})
        print('SKIP invalid_video:', path)
        continue
    if not prep_frames:
        skipped_videos.append({'asset_id': asset_id, 'local_path': path.as_posix(), 'error': 'no frames extracted'})
        print('SKIP no_frames:', path)
        continue
    first = cv2.imread(str(raw_frames[0])) if raw_frames else None
    lighting_cv, brightness = lighting_from_frame(first) if first is not None else ('unknown', 0.0)
    video_rows.append({
        'coarse_label': CATEGORY_LABEL,
        'asset_id': asset_id,
        'local_path': path.as_posix(),
        'duration_sec': video_duration_sec(path),
        'frame_dir': raw_frame_dir.as_posix(),
        'preprocess_frame_dir': prep_frame_dir.as_posix(),
        'frame_count': len(prep_frames),
        'lighting_cv': lighting_cv,
        'brightness_mean': round(brightness, 3),
        'is_night': str(lighting_cv in {'night', 'low_light'}),
    })

print('video_rows:', len(video_rows))
print('skipped_videos:', len(skipped_videos))
for row in skipped_videos[:10]:
    print(row)
print('lighting_cv:', dict(Counter(row['lighting_cv'] for row in video_rows)))


In [ ]:
BOX_COLORS = {
    'person': (255, 60, 60),
    'bicycle': (80, 220, 120),
    'car': (60, 150, 255),
    'motorcycle': (255, 180, 60),
    'bus': (220, 80, 255),
    'truck': (255, 255, 80),
}


def draw_boxes(image_path, boxes, names, out_path):
    image = PILImage.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    for box in boxes:
        cls_name = names[int(box.cls[0])]
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(float, box.xyxy[0].tolist())
        color = BOX_COLORS.get(cls_name, (255, 255, 255))
        draw.rectangle([x1, y1, x2, y2], outline=color, width=8)
        draw.rectangle([x1, max(0, y1 - 24), x1 + 150, y1], fill=(0, 0, 0))
        draw.text((x1 + 4, max(0, y1 - 22)), f'{cls_name} {conf:.2f}', fill=color)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(out_path, quality=90)
    return out_path


def make_contact_sheet(image_paths, out_path, title='', cols=4, thumb_w=420):
    images = [PILImage.open(p).convert('RGB') for p in image_paths]
    if not images:
        return None
    thumb_h = int(images[0].height * thumb_w / images[0].width)
    rows = math.ceil(len(images) / cols)
    title_h = 40 if title else 0
    sheet = PILImage.new('RGB', (cols * thumb_w, rows * thumb_h + title_h), 'white')
    draw = ImageDraw.Draw(sheet)
    if title:
        draw.text((8, 8), title, fill='black')
    for i, image in enumerate(images):
        image.thumbnail((thumb_w, thumb_h))
        x = (i % cols) * thumb_w
        y = title_h + (i // cols) * thumb_h
        sheet.paste(image, (x, y))
        draw.text((x + 4, y + 4), str(i), fill='yellow')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(out_path, quality=90)
    return out_path


## YOLO 5개 모델 비교

OpenCV 전처리 프레임에 대해 YOLO 모델 5개를 동일 조건으로 실행한다.


In [ ]:
summary_rows = []
detail_rows = []

for model_name in YOLO_MODELS:
    print('## model:', model_name)
    model = YOLO(model_name)
    model_key = model_name.replace('.pt', '')
    for row in tqdm(video_rows, desc=model_name):
        frames = sorted(Path(row['preprocess_frame_dir']).glob('frame_*.jpg'))
        annotated_paths = []
        video_box_count = 0
        video_relevant_box_count = 0
        frames_with_detection = 0
        frames_with_relevant_detection = 0
        confs = []
        class_counter = Counter()

        for frame_path in frames:
            result = model.predict(str(frame_path), conf=YOLO_CONF, imgsz=YOLO_IMGSZ, verbose=False)[0]
            boxes = list(result.boxes) if result.boxes is not None else []
            relevant_boxes = []
            for box in boxes:
                cls_name = result.names[int(box.cls[0])]
                conf = float(box.conf[0])
                class_counter[cls_name] += 1
                confs.append(conf)
                if cls_name in RELEVANT_CLASSES:
                    relevant_boxes.append(box)
            if boxes:
                frames_with_detection += 1
            if relevant_boxes:
                frames_with_relevant_detection += 1
            video_box_count += len(boxes)
            video_relevant_box_count += len(relevant_boxes)

            ann_path = OUTPUT_DIR / 'annotated_frames' / model_key / row['asset_id'] / frame_path.name
            draw_boxes(frame_path, relevant_boxes, result.names, ann_path)
            annotated_paths.append(ann_path)

        contact_path = CONTACT_DIR / model_key / f"{row['asset_id']}_contact.jpg"
        make_contact_sheet(annotated_paths, contact_path, title=f"{model_name} | {CATEGORY_LABEL} | {row['asset_id']} | {row['lighting_cv']}")
        summary_rows.append({
            **row,
            'yolo_model': model_name,
            'frames_with_detection': frames_with_detection,
            'frames_with_relevant_detection': frames_with_relevant_detection,
            'total_box_count': video_box_count,
            'total_relevant_box_count': video_relevant_box_count,
            'avg_conf': round(sum(confs) / len(confs), 4) if confs else 0,
            'class_counts': json.dumps(dict(class_counter), ensure_ascii=False),
            'contact_sheet_path': contact_path.as_posix(),
        })

print('summary_rows:', len(summary_rows))


In [ ]:
def write_csv(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    fields = list(dict.fromkeys(key for row in rows for key in row.keys()))
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    with temporary_path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary_path.replace(path)

write_csv(summary_rows, SUMMARY_CSV)
write_csv(detail_rows, DETAIL_CSV)
print('SUMMARY_CSV:', SUMMARY_CSV)


In [ ]:
print('## YOLO model summary')
for model_name in YOLO_MODELS:
    rows = [r for r in summary_rows if r['yolo_model'] == model_name]
    if not rows:
        continue
    print(model_name)
    print(' videos:', len(rows))
    print(' avg relevant frames:', round(sum(r['frames_with_relevant_detection'] for r in rows) / len(rows), 2))
    print(' avg relevant boxes:', round(sum(r['total_relevant_box_count'] for r in rows) / len(rows), 2))
    print(' avg conf:', round(sum(r['avg_conf'] for r in rows) / len(rows), 4))

# contact sheet preview
for row in summary_rows[:min(10, len(summary_rows))]:
    print(row['yolo_model'], row['asset_id'], row['lighting_cv'])
    display(Image(filename=row['contact_sheet_path']))


## Qwen2.5-VL 비교

YOLO/contact sheet는 100프레임 기준으로 유지하되, VLM에는 설정된 수의 대표 프레임만 균등 샘플링해 전달한다.


In [ ]:
import gc
import json
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

QWEN_MODEL_ID = EXPERIMENT.qwen_model_id
from ai.vision.vlm_json import VLM_JSON_PROMPT, VLM_JSON_RETRY_PROMPT, parse_vlm_json

def clear_gpu_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def extract_json_object(text):
    return parse_vlm_json(text)

def frame_number(path):
    try:
        return int(Path(path).stem.split('_')[1])
    except Exception:
        return 0


def sample_evenly(paths, count):
    if count < 1:
        raise ValueError('frame sample count must be positive')
    paths = list(paths)
    if len(paths) <= count:
        return paths
    return [paths[round(i * (len(paths) - 1) / (count - 1))] for i in range(count)]


def annotated_frame_paths(row):
    model_key = row['yolo_model'].replace('.pt', '')
    frame_dir = OUTPUT_DIR / 'annotated_frames' / model_key / row['asset_id']
    return sorted(frame_dir.glob('frame_*.jpg'), key=frame_number)


def resolve_accident_target(predicted_target, user_target=None):
    predicted = str(predicted_target or '').strip().lower()
    user = str(user_target or '').strip().lower()
    valid = {'car_vs_car', 'car_vs_pedestrian', 'car_vs_motorcycle', 'car_vs_bicycle'}
    final = predicted if predicted in valid else user if user in valid else 'uncertain'
    return {
        'final_accident_target': final,
        'needs_user_input': final == 'uncertain',
        'user_question': '사고 상대는 차량, 보행자, 이륜차, 자전거 중 무엇인가요?' if final == 'uncertain' else '',
    }


def qwen_analyze_images(image_paths, model, processor):
    if not image_paths:
        raise ValueError('Qwen input frames are missing.')
    used_paths = select_collision_aware_frames(image_paths, QWEN_INPUT_FRAME_COUNT)
    last_output, last_error = '', 'json_incomplete:no_attempt'
    for prompt_text in (VLM_JSON_PROMPT, VLM_JSON_RETRY_PROMPT):
        try:
            content = [{'type': 'image', 'image': str(path), 'max_pixels': 640 * 360} for path in used_paths]
            content.append({'type': 'text', 'text': prompt_text})
            messages = [{'role': 'user', 'content': content}]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(model.device)
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False, use_cache=False)
            last_output = processor.batch_decode(
                generated_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
            )[0]
            parsed, valid, last_error = parse_vlm_json(last_output)
            del inputs, generated_ids
            clear_gpu_cache()
            if valid:
                return parsed, last_output, True, '', len(used_paths)
        except torch.cuda.OutOfMemoryError:
            clear_gpu_cache()
            return {}, last_output, False, 'CUDA OOM with required frame count', 0
    return {}, last_output, False, last_error, len(used_paths)


In [ ]:
# 지정 모델이 없으면 영상별 탐지 성능이 가장 높은 YOLO 결과를 선택한다.
YOLO_QWEN_MODEL = EXPERIMENT.qwen_yolo_model
if YOLO_QWEN_MODEL:
    if YOLO_QWEN_MODEL not in YOLO_MODELS:
        raise ValueError(f'VISION_QWEN_YOLO_MODEL is not in VISION_YOLO_MODELS: {YOLO_QWEN_MODEL}')
    best_rows = [row for row in summary_rows if row['yolo_model'] == YOLO_QWEN_MODEL]
else:
    best_rows = []
    for asset_id in sorted({row['asset_id'] for row in summary_rows}):
        candidates = [row for row in summary_rows if row['asset_id'] == asset_id]
        best_rows.append(max(candidates, key=lambda row: (row['frames_with_relevant_detection'], row['avg_conf'])))

if not best_rows:
    raise RuntimeError('Qwen 입력 대상이 없습니다. YOLO 셀을 먼저 실행하세요.')
print('YOLO_QWEN_MODEL:', YOLO_QWEN_MODEL or 'best_per_video')
print('best_rows:', len(best_rows))


In [ ]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID, torch_dtype='auto', device_map='auto'
)
processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)

from ai.vision.trained_category_classifier import TrainedCategoryClassifier, find_best_checkpoint

TRAINED_CHECKPOINT = find_best_checkpoint(PROJECT_ROOT)
TRAINED_CLASSIFIER = TrainedCategoryClassifier(TRAINED_CHECKPOINT)
print('TRAINED_CHECKPOINT:', TRAINED_CHECKPOINT)


def apply_trained_target(row):
    row['vlm_predicted_accident_target'] = row.get('vlm_predicted_accident_target') or row.get('predicted_accident_target', 'uncertain')
    trained = TRAINED_CLASSIFIER.predict(row['source_video_path'])
    row['labeled_accident_target'] = CATEGORY_KEY
    row['predicted_accident_target'] = trained['label']
    row['trained_prediction_confidence'] = trained['confidence']
    row['trained_classifier_checkpoint'] = trained['checkpoint']
    return row


qwen_results = []
if QWEN_OUTPUT_CSV.exists():
    with QWEN_OUTPUT_CSV.open(encoding='utf-8-sig', newline='') as f:
        qwen_results = list(csv.DictReader(f))
qwen_retry_ids = {row['asset_id'] for row in qwen_results if str(row.get('qwen_json_valid', '')).lower() != 'true'}
completed_ids = {row['asset_id'] for row in qwen_results} - qwen_retry_ids
print('qwen_retry_pending:', len(qwen_retry_ids))
print('resume_completed:', len(completed_ids), '/', len(best_rows))

for index, row in enumerate(best_rows, 1):
    if row['asset_id'] in completed_ids:
        continue
    try:
        parsed, raw_output, valid, error, used_frame_count = qwen_analyze_images(
            annotated_frame_paths(row), model, processor
        )
    except Exception as exc:
        clear_gpu_cache()
        parsed, raw_output, valid, error, used_frame_count = {}, '', False, repr(exc), 0
    decision = resolve_accident_target(parsed.get('predicted_accident_target'))
    result = apply_trained_target({
        'coarse_label': row.get('coarse_label', CATEGORY_LABEL),
        'asset_id': row['asset_id'],
        'source_video_path': row['local_path'],
        'yolo_model': row['yolo_model'],
        'lighting_cv': row['lighting_cv'],
        'is_night': row['is_night'],
        'contact_sheet_path': row['contact_sheet_path'],
        'qwen_input_frame_count': used_frame_count,
        'qwen_json_valid': str(valid),
        'parse_error': error,
        'predicted_accident_target': decision['final_accident_target'],
        'accident_target_evidence': parsed.get('accident_target_evidence', ''),
        'needs_user_input': str(decision['needs_user_input']),
        'user_question': decision['user_question'],
        'accident_visible': parsed.get('accident_visible', ''),
        'accident_visibility': parsed.get('accident_visibility', ''),
        'collision_moment_visible': parsed.get('collision_moment_visible', ''),
        'bbox_helpfulness': parsed.get('bbox_helpfulness', ''),
        'bbox_quality': parsed.get('bbox_quality', ''),
        'scene_conditions': json.dumps(parsed.get('scene_conditions', {}), ensure_ascii=False),
        'visible_objects': json.dumps(parsed.get('visible_objects', []), ensure_ascii=False),
        'uncertainties': json.dumps(parsed.get('uncertainties', []), ensure_ascii=False),
        'summary': parsed.get('summary', ''),
        'accident_situation': parsed.get('accident_situation', ''),
        'raw_output_text': raw_output,
    })
    qwen_results = [existing for existing in qwen_results if existing.get('asset_id') != row['asset_id']]
    qwen_results.append(result)
    write_csv(qwen_results, QWEN_OUTPUT_CSV)
    print(f"[{index}/{len(best_rows)}]", row['asset_id'], result['predicted_accident_target'], error)

print('QWEN_OUTPUT_CSV:', QWEN_OUTPUT_CSV)
print('qwen_result_count:', len(qwen_results))


In [ ]:
from IPython.display import Video, display

def is_qwen_missed(row):
    visibility = str(row.get('accident_visibility', '')).strip().lower()
    collision = str(row.get('collision_moment_visible', '')).strip().lower()
    visible = str(row.get('accident_visible', '')).strip().lower()
    return not (visibility == 'clear' and collision == 'true' and visible == 'true')

# 사고 데이터라는 정답과 달리 Qwen이 명확한 충돌을 찾지 못한 결과를 전부 저장하고 재생한다.
missed_rows = [row for row in qwen_results if is_qwen_missed(row)]
QWEN_MISSED_CSV = OUTPUT_DIR / 'qwen_missed_results.csv'
write_csv(missed_rows, QWEN_MISSED_CSV)
print('qwen_missed_count:', len(missed_rows))
print('QWEN_MISSED_CSV:', QWEN_MISSED_CSV)

for row in missed_rows:
    print('\n##', row['asset_id'])
    print('predicted_accident_target:', row.get('predicted_accident_target'))
    print('accident_visible:', row.get('accident_visible'))
    print('accident_visibility:', row.get('accident_visibility'))
    print('collision_moment_visible:', row.get('collision_moment_visible'))
    print('summary:', row.get('summary'))
    print('accident_situation:', row.get('accident_situation'))
    source_video = Path(row['source_video_path'])
    print('source_video:', source_video)
    if source_video.exists():
        display(Video(str(source_video), embed=False, width=640))


## 사고 탐지 VLM 비교: Qwen2.5-VL vs LLaVA-OneVision

동일한 500개 사고 영상, 동일한 YOLO 프레임, 동일한 프롬프트와 판정 기준으로 비교한다.

| 모델 | 간략한 설명 | 장점 | 단점 |
|---|---|---|---|
| Qwen2.5-VL-3B-Instruct | 이미지·영상의 시간적/공간적 내용을 텍스트로 설명하는 경량 Vision-Language Model | 비교적 적은 GPU 메모리, 한국어 출력과 JSON 지시 이행이 양호, 현재 파이프라인 검증 완료 | 3B 모델이라 복잡하거나 짧은 충돌 장면을 놓칠 수 있음 |
| LLaVA-OneVision-Qwen2-7B-OV | 단일 이미지, 다중 이미지, 영상을 함께 다루도록 학습된 7B급 Vision-Language Model | 더 큰 모델 용량, 여러 프레임을 연결한 장면 이해에 유리 | Qwen 3B보다 느리고 GPU 메모리를 더 사용하며 한국어/JSON 형식이 흔들릴 수 있음 |

### 선택 지표

- **사고 탐지율** = `accident_visible=true`, `collision_moment_visible=true`, `accident_visibility=clear`를 모두 만족한 영상 수 / 전체 처리 영상 수
- 모든 입력이 사고 영상이므로 이 값은 사고 양성 데이터에 대한 **재현율(recall) 성격의 지표**이다.
- 모델 오류나 JSON 파싱 실패는 탐지 성공으로 계산하지 않는다.
- 정상 영상이 없으므로 오탐률과 정밀도는 평가할 수 없다. 1차 모델 선택은 사고 탐지율을 우선하고, 동률이면 JSON 성공률과 작은 모델을 우선한다.

### 사고 대상 처리 원칙

정답 `coarse_label`은 평가에만 사용하고 모델 프롬프트에는 넣지 않는다. 모델이 `predicted_accident_target`을 먼저 예측하며, `uncertain`일 때만 사용자에게 사고 상대를 질문한다.


In [ ]:
from PIL import Image as PILImage
from transformers import LlavaOnevisionForConditionalGeneration

LLAVA_MODEL_ID = EXPERIMENT.llava_model_id
LLAVA_OUTPUT_CSV = OUTPUT_DIR / 'llava_onevision_results.csv'

if 'model' in globals():
    del model
if 'processor' in globals():
    del processor
clear_gpu_cache()
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    LLAVA_MODEL_ID, torch_dtype='auto', device_map='auto', low_cpu_mem_usage=True
)
processor = AutoProcessor.from_pretrained(LLAVA_MODEL_ID)


def llava_analyze_images(image_paths):
    used_paths = sample_evenly(image_paths, QWEN_INPUT_FRAME_COUNT)
    if not used_paths:
        raise ValueError('LLaVA input frames are missing.')
    images = [PILImage.open(path).convert('RGB') for path in used_paths]
    last_output, last_error = '', 'json_incomplete:no_attempt'
    try:
        for prompt_text in (VLM_JSON_PROMPT, VLM_JSON_RETRY_PROMPT):
            content = [{'type': 'image'} for _ in images]
            content.append({'type': 'text', 'text': prompt_text})
            messages = [{'role': 'user', 'content': content}]
            prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
            inputs = processor(images=images, text=prompt, return_tensors='pt').to(model.device)
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
            last_output = processor.batch_decode(
                generated_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
            )[0]
            parsed, valid, last_error = parse_vlm_json(last_output)
            del inputs, generated_ids
            clear_gpu_cache()
            if valid:
                return parsed, last_output, True, '', len(used_paths)
        return {}, last_output, False, last_error, len(used_paths)
    finally:
        for image in images:
            image.close()

llava_results = []
if LLAVA_OUTPUT_CSV.exists():
    with LLAVA_OUTPUT_CSV.open(encoding='utf-8-sig', newline='') as f:
        llava_results = list(csv.DictReader(f))
llava_retry_ids = {row['asset_id'] for row in llava_results if str(row.get('json_valid', '')).lower() != 'true'}
completed_ids = {row['asset_id'] for row in llava_results} - llava_retry_ids
print('llava_retry_pending:', len(llava_retry_ids))
print('llava_resume_completed:', len(completed_ids), '/', len(best_rows))

for index, row in enumerate(best_rows, 1):
    if row['asset_id'] in completed_ids:
        continue
    try:
        parsed, raw_output, valid, error, used_frame_count = llava_analyze_images(annotated_frame_paths(row))
    except Exception as exc:
        clear_gpu_cache()
        parsed, raw_output, valid, error, used_frame_count = {}, '', False, repr(exc), 0
    decision = resolve_accident_target(parsed.get('predicted_accident_target'))
    result = apply_trained_target({
        'coarse_label': row.get('coarse_label', CATEGORY_LABEL),
        'asset_id': row['asset_id'],
        'source_video_path': row['local_path'],
        'yolo_model': row['yolo_model'],
        'model_id': LLAVA_MODEL_ID,
        'input_frame_count': used_frame_count,
        'json_valid': str(valid),
        'parse_error': error,
        'predicted_accident_target': decision['final_accident_target'],
        'accident_target_evidence': parsed.get('accident_target_evidence', ''),
        'needs_user_input': str(decision['needs_user_input']),
        'user_question': decision['user_question'],
        'accident_visible': parsed.get('accident_visible', ''),
        'accident_visibility': parsed.get('accident_visibility', ''),
        'collision_moment_visible': parsed.get('collision_moment_visible', ''),
        'bbox_helpfulness': parsed.get('bbox_helpfulness', ''),
        'bbox_quality': parsed.get('bbox_quality', ''),
        'scene_conditions': json.dumps(parsed.get('scene_conditions', {}), ensure_ascii=False),
        'visible_objects': json.dumps(parsed.get('visible_objects', []), ensure_ascii=False),
        'uncertainties': json.dumps(parsed.get('uncertainties', []), ensure_ascii=False),
        'summary': parsed.get('summary', ''),
        'accident_situation': parsed.get('accident_situation', ''),
        'raw_output_text': raw_output,
    })
    llava_results = [existing for existing in llava_results if existing.get('asset_id') != row['asset_id']]
    llava_results.append(result)
    write_csv(llava_results, LLAVA_OUTPUT_CSV)
    print(f"[{index}/{len(best_rows)}]", row['asset_id'], result['predicted_accident_target'], error)

print('LLAVA_OUTPUT_CSV:', LLAVA_OUTPUT_CSV)


In [ ]:
# LLaVA가 명확한 사고 장면을 찾지 못한 영상을 제한 없이 전부 저장하고 재생한다.
LLAVA_MISSED_CSV = OUTPUT_DIR / 'llava_missed_results.csv'
llava_missed_rows = [row for row in llava_results if is_qwen_missed(row)]
write_csv(llava_missed_rows, LLAVA_MISSED_CSV)
print('llava_missed_count:', len(llava_missed_rows))
print('LLAVA_MISSED_CSV:', LLAVA_MISSED_CSV)

for row in llava_missed_rows:
    print('\n##', row['asset_id'])
    print('predicted_accident_target:', row.get('predicted_accident_target'))
    print('accident_visible:', row.get('accident_visible'))
    print('accident_visibility:', row.get('accident_visibility'))
    print('collision_moment_visible:', row.get('collision_moment_visible'))
    print('summary:', row.get('summary'))
    print('accident_situation:', row.get('accident_situation'))
    source_video = Path(row['source_video_path'])
    print('source_video:', source_video)
    if source_video.exists():
        display(Video(str(source_video), embed=False, width=640))


In [ ]:
VLM_COMPARE_OUTPUT_CSV = OUTPUT_DIR / 'vlm_accident_detection_comparison.csv'
LABEL_TO_TARGET = {
    '차대차': 'car_vs_car',
    '차대보행자': 'car_vs_pedestrian',
    '차대이륜차': 'car_vs_motorcycle',
    '차대자전거': 'car_vs_bicycle',
}


def is_accident_detected(row):
    return (
        str(row.get('accident_visible', '')).strip().lower() == 'true'
        and str(row.get('collision_moment_visible', '')).strip().lower() == 'true'
        and str(row.get('accident_visibility', '')).strip().lower() == 'clear'
    )


def model_metrics(model_name, rows, valid_key):
    total = len(rows)
    detected = sum(is_accident_detected(row) for row in rows)
    valid = sum(str(row.get(valid_key, '')).strip().lower() == 'true' for row in rows)
    target_correct = sum(
        row.get('predicted_accident_target') == LABEL_TO_TARGET.get(row.get('coarse_label'))
        for row in rows
    )
    return {
        'model': model_name,
        'processed_count': total,
        'detected_count': detected,
        'accident_detection_rate': round(detected / total, 4) if total else 0,
        'target_correct_count': target_correct,
        'target_accuracy': round(target_correct / total, 4) if total else 0,
        'json_valid_count': valid,
        'json_valid_rate': round(valid / total, 4) if total else 0,
    }


comparison_rows = [
    model_metrics(QWEN_MODEL_ID, qwen_results, 'qwen_json_valid'),
    model_metrics(LLAVA_MODEL_ID, llava_results, 'json_valid'),
]
write_csv(comparison_rows, VLM_COMPARE_OUTPUT_CSV)
display(comparison_rows)
print('VLM_COMPARE_OUTPUT_CSV:', VLM_COMPARE_OUTPUT_CSV)

eligible = [row for row in comparison_rows if row['processed_count'] == len(best_rows)]
if len(eligible) == 2:
    recommended = max(
        eligible,
        key=lambda row: (row['accident_detection_rate'], row['target_accuracy'], row['json_valid_rate'])
    )
    print('recommended_model:', recommended['model'])
else:
    print('두 모델 모두 전체 영상을 처리한 뒤 최종 모델을 선택하세요.')


def _logic_self_check():
    assert resolve_accident_target('car_vs_car')['needs_user_input'] is False
    assert resolve_accident_target('uncertain')['needs_user_input'] is True
    rows = [
        {'coarse_label': '차대차', 'predicted_accident_target': 'car_vs_car', 'accident_visible': True,
         'collision_moment_visible': 'true', 'accident_visibility': 'clear', 'ok': 'true'},
        {'coarse_label': '차대차', 'predicted_accident_target': 'uncertain', 'accident_visible': 'false',
         'collision_moment_visible': 'true', 'accident_visibility': 'clear', 'ok': 'true'},
    ]
    result = model_metrics('test', rows, 'ok')
    assert result['accident_detection_rate'] == 0.5 and result['target_accuracy'] == 0.5


_logic_self_check()
